# Part 4: Gradient Descent in Code

This notebook converts the manual gradient descent calculations from **Part 3** into Python.
Every step mirrors the manual arithmetic exactly (same X, y, initial m, b, and learning rate),
so the printed values at each iteration should match the handwritten calculations.

**Contents**
1. Setup (data, initial parameters)
2. Prediction function
3. Cost (MSE) function
4. Gradient — manual (closed-form) derivation
5. Gradient — verified with SciPy's numerical derivative
6. Gradient Descent loop (step-by-step, not abstracted)
7. Visualizations: parameter convergence + error convergence


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import approx_fprime

np.set_printoptions(precision=4, suppress=True)


## 1. Setup

Same values used in the manual (Part 3) calculations:

- Model: ŷ = Xm + b
- X = [[1, 3], [4, 10]]
- y = [5, 6]
- Initial m = [-1, 2]
- Initial b = [1, 1]
- Learning rate α = 0.01
- Iterations = 4 (one per group member)


In [ ]:
X = np.array([[1, 3],
              [4, 10]], dtype=float)

y = np.array([5, 6], dtype=float)

m = np.array([-1.0, 2.0])   # initial m
b = np.array([1.0, 1.0])    # initial b

alpha = 0.01
n_iterations = 4
n = len(y)

print("X =\n", X)
print("y =", y)
print("Initial m =", m)
print("Initial b =", b)


## 2. Predicted Values

ŷ = Xm + b, computed with explicit matrix multiplication (`X @ m`) — not hidden inside a
black-box `.fit()` call.


In [ ]:
def predict(X, m, b):
    """Compute y_hat = X @ m + b, matching the manual row-by-row calculation."""
    return X @ m + b

y_hat0 = predict(X, m, b)
print("y_hat0 =", y_hat0)   # Expect [6, 17], matching Step 1 of the manual calculations


## 3. Error and Cost (MSE)

J(m, b) = (1/n) * Σ(e_i²), where e = ŷ - y


In [ ]:
def compute_error(y_hat, y):
    return y_hat - y

def compute_cost(y_hat, y):
    e = compute_error(y_hat, y)
    return (1 / len(y)) * np.sum(e ** 2)

e0 = compute_error(y_hat0, y)
J0 = compute_cost(y_hat0, y)
print("e0 =", e0)   # Expect [1, 11]
print("J0 =", J0)   # Expect 61.0


## 4. Gradient — Manual (Closed-Form) Derivation

From the chain rule (Part 3, Step 3):

- dJ/dm = (2/n) * Xᵀ · e
- dJ/db = (2/n) * e

Since n = 2 here, (2/n) = 1, so dJ/dm = Xᵀ·e and dJ/db = e for this dataset.


In [ ]:
def gradient_m(X, e, n):
    """dJ/dm = (2/n) * X^T . e"""
    return (2 / n) * (X.T @ e)

def gradient_b(e, n):
    """dJ/db = (2/n) * e"""
    return (2 / n) * e

dJdm_manual = gradient_m(X, e0, n)
dJdb_manual = gradient_b(e0, n)
print("dJ/dm (manual) =", dJdm_manual)   # Expect [45, 113]
print("dJ/db (manual) =", dJdb_manual)   # Expect [1, 11]


## 5. Gradient — Verified with SciPy

Using `scipy.optimize.approx_fprime`, we numerically differentiate the cost function J(m, b)
with respect to the parameters. This confirms our hand-derived formula above is correct,
independent of the closed-form derivation.


In [ ]:
def cost_given_params(params, X, y):
    """
    Flattened cost function J(params) so SciPy can differentiate it.
    params = [m1, m2, b1, b2]
    """
    m_ = params[:2]
    b_ = params[2:]
    y_hat = X @ m_ + b_
    e = y_hat - y
    return (1 / len(y)) * np.sum(e ** 2)

params0 = np.concatenate([m, b])          # [-1, 2, 1, 1]
epsilon = np.sqrt(np.finfo(float).eps)

grad_numeric = approx_fprime(params0, cost_given_params, epsilon, X, y)

dJdm_scipy = grad_numeric[:2]
dJdb_scipy = grad_numeric[2:]

print("dJ/dm (SciPy numerical) =", dJdm_scipy)
print("dJ/db (SciPy numerical) =", dJdb_scipy)
print()
print("Matches manual derivation:",
      np.allclose(dJdm_scipy, dJdm_manual, atol=1e-4) and
      np.allclose(dJdb_scipy, dJdb_manual, atol=1e-4))


## 6. Gradient Descent Loop

Each iteration is shown explicitly: recompute predictions → error → cost → gradient → update.
Nothing is hidden inside a single `.fit()` call, so every step's numbers can be checked
against the handwritten Part 3 calculations.


In [ ]:
# Re-initialize so this cell can be re-run independently
m = np.array([-1.0, 2.0])
b = np.array([1.0, 1.0])

history = {
    "m1": [m[0]], "m2": [m[1]],
    "b1": [b[0]], "b2": [b[1]],
    "cost": []
}

for i in range(n_iterations):
    # 1. Predict
    y_hat = predict(X, m, b)

    # 2. Error and cost
    e = compute_error(y_hat, y)
    J = compute_cost(y_hat, y)
    history["cost"].append(J)

    # 3. Gradients
    dJdm = gradient_m(X, e, n)
    dJdb = gradient_b(e, n)

    # 4. Parameter update
    m = m - alpha * dJdm
    b = b - alpha * dJdb

    # Record post-update parameters
    history["m1"].append(m[0])
    history["m2"].append(m[1])
    history["b1"].append(b[0])
    history["b2"].append(b[1])

    print(f"Iteration {i+1}:")
    print(f"  y_hat = {y_hat}")
    print(f"  e     = {e}")
    print(f"  J     = {J:.4f}")
    print(f"  dJ/dm = {dJdm}")
    print(f"  dJ/db = {dJdb}")
    print(f"  m     = {m}")
    print(f"  b     = {b}")
    print()

# Final cost after the last update, for the summary table
y_hat_final = predict(X, m, b)
J_final = compute_cost(y_hat_final, y)
history["cost"].append(J_final)

print("Final m:", m)
print("Final b:", b)
print("Final cost:", J_final)


### Summary Table

Cross-check against the manual Part 3 summary table.


In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Iteration": range(n_iterations + 1),
    "m1": history["m1"],
    "m2": history["m2"],
    "b1": history["b1"],
    "b2": history["b2"],
    "Cost J": history["cost"],
})
summary.round(4)


## 7. Visualizations

### Plot 1 — How m and b change over iterations


In [ ]:
iterations = range(n_iterations + 1)

plt.figure(figsize=(8, 5))
plt.plot(iterations, history["m1"], marker="o", label="m1")
plt.plot(iterations, history["m2"], marker="o", label="m2")
plt.plot(iterations, history["b1"], marker="s", label="b1")
plt.plot(iterations, history["b2"], marker="s", label="b2")
plt.xlabel("Iteration")
plt.ylabel("Parameter value")
plt.title("Parameter Convergence (m and b) over Iterations")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("parameters_over_iterations.png", dpi=150)
plt.show()


### Plot 2 — How the error (cost J) changes over iterations


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(iterations, history["cost"], marker="o", color="crimson")
plt.xlabel("Iteration")
plt.ylabel("Cost J (MSE)")
plt.title("Cost Function Convergence over Iterations")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("cost_over_iterations.png", dpi=150)
plt.show()


## Conclusion

The cost drops sharply after the first update (61.0 → ~6.5) and then continues to fall by
smaller amounts each iteration, which is the expected shape of gradient descent: large
corrective steps early on, and progressively smaller refinements as the parameters approach
a minimum. The values computed here match the manual calculations in Part 3, and the SciPy
numerical derivative confirms the hand-derived gradient formulas were correct.
